# S0-3 DNI*cosZ Test - PLTS-IKN

Library-first Colab/local runner for Sprint 0 discovery. The notebook stages source files onto the local VM, then delegates ingestion, alignment, statistics, sensitivity analysis, plotting, and the deterministic metadata decision to `src/characterisation/`. It does not build a forecasting model.

In [ ]:
import os

REPOSITORY_URL = os.environ.get('S03_REPOSITORY_URL', 'https://github.com/ompltsikn/Forecasting-Irradiance.git')
REPO_ROOT = os.environ.get('S03_REPO_ROOT', '/content/Forecasting-Irradiance')
DRIVE_RAW_DATA_DIR = os.environ.get('S03_DRIVE_RAW_DATA_DIR', '/content/drive/Shareddrives/Forecasting Irradiance/raw_data')
DRIVE_HISTORICAL_XLSX_ROOT = os.environ.get('S03_DRIVE_HISTORICAL_XLSX_ROOT', '/content/drive/Shareddrives/Forecasting Irradiance/Data Weather Station')
LOCAL_STAGE_ROOT = os.environ.get('S03_LOCAL_STAGE_ROOT', '/content/s03_local_stage')
OUTPUT_DIR = os.environ.get('S03_OUTPUT_DIR', '/content/s03_output')
TAG_STATS = os.environ.get('S03_TAG_STATS', f'{REPO_ROOT}/artifacts/phase0_cov/tag_characterisation.csv')
SITE_CONFIG = os.environ.get('S03_SITE_CONFIG', f'{REPO_ROOT}/configs/site_plts-ikn.yaml')
STRICT_MODE = os.environ.get('S03_STRICT_MODE', 'true').strip().lower() in {'1', 'true', 'yes'}
SKIP_DRIVE_MOUNT = os.environ.get('S03_SKIP_DRIVE_MOUNT', 'false').strip().lower() in {'1', 'true', 'yes'}

In [ ]:
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
repo_path = Path(REPO_ROOT).resolve()
if not repo_path.is_dir():
    if not IN_COLAB:
        raise FileNotFoundError(f'Repository not found: {repo_path}')
    subprocess.run(['git', 'clone', '--depth', '1', REPOSITORY_URL, str(repo_path)], check=True)
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(repo_path / 'requirements-cov.txt')], check=True)
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))
print({'runtime': 'colab' if IN_COLAB else 'local', 'repo_root': str(repo_path)})

In [ ]:
if IN_COLAB and not SKIP_DRIVE_MOUNT:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Drive mount skipped; using configured filesystem paths.')

In [ ]:
from src.characterisation.dni_cosz_cli import run_dni_cosz_test, stage_dni_cosz_inputs

staged = stage_dni_cosz_inputs(
    raw_source_dir=Path(DRIVE_RAW_DATA_DIR),
    historical_source_root=Path(DRIVE_HISTORICAL_XLSX_ROOT),
    local_stage_root=Path(LOCAL_STAGE_ROOT),
)
print({'staged_file_count': staged.file_count, 'staged_bytes': staged.byte_count, 'sha256_verified': staged.sha256_verified})
result = run_dni_cosz_test(
    raw_dir=staged.raw_dir,
    historical_xlsx_root=staged.historical_root,
    output_dir=Path(OUTPUT_DIR),
    tag_stats_path=Path(TAG_STATS),
    site_config_path=Path(SITE_CONFIG),
    strict=STRICT_MODE,
)
if STRICT_MODE and result.strict_errors:
    raise RuntimeError(f'Strict S0-3 gate failed after diagnostics: {result.strict_errors}')
print(result.summary)

In [ ]:
from IPython.display import Image, JSON, display

display(JSON(result.summary))
display(Image(filename=str(result.artifacts.figure_path)))